In [1]:
from langchain_community.document_loaders import PyPDFLoader

C:\Users\Asfar Butt\AppData\Local\Temp\ipykernel_11380\4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
e:\Asfar\Learning\LangChain\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
loader = PyPDFLoader("Sophisticated_Topic_Transformer_Architecture.pdf")

doc = loader.load()

print(doc[0].page_content)

Transformer Architecture in Modern AI
Overview
The Transformer architecture, introduced in 2017 by the paper Attention Is All You Need,
revolutionized machine learning by replacing recurrent neural networks with a self-attention
mechanism. This enabled parallel processing of sequences, dramatically improving training
efficiency and scaling.
Core Components
1. Tokenization and Embeddings: Text is split into tokens and converted into dense vectors.
2. Positional Encoding: Since attention is permutation invariant, positional information is injected
into embeddings.
3. Multi-Head Self-Attention: Each token attends to every other token, learning contextual
relationships through Query, Key, and Value projections.
4. Feed-Forward Networks: Nonlinear transformations applied independently to each token
representation.
5. Residual Connections and Layer Normalization: Improve gradient flow and stabilize optimization.
Attention Mechanism
Scaled dot-product attention computes: Attention(Q,K,V)=soft

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
splitter = RecursiveCharacterTextSplitter(
            chunk_size=100,
            chunk_overlap=20)

chunks = splitter.split_documents(doc)

print(chunks)

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-25T10:15:24+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-07-25T10:15:24+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'Sophisticated_Topic_Transformer_Architecture.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Transformer Architecture in Modern AI\nOverview'), Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-07-25T10:15:24+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-07-25T10:15:24+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'Sophisticated_Topic_Transformer_Architecture.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Overview\nThe Transformer architecture, introduced in 2017 by the paper Attention Is All You Need,'), 

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

e:\Asfar\Learning\LangChain\.venv\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asfar Butt\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:01<00:00, 89.34it/s]


In [11]:
from langchain_community.vectorstores import Chroma

In [12]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="my_database"
)

In [13]:
retriever = vector_store.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k": 2}
)

In [17]:
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()

In [15]:
from langchain_core.runnables import RunnablePassthrough

In [22]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
    """
    You are an AI asistant and your job is to review the following context
    Context:{context}
    and give a detailed answer to the following question
     Question:{question}
""")

In [49]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY
)

print(GROQ_API_KEY[:8])
print(GROQ_API_KEY[-8:])
print(len(GROQ_API_KEY))

gsk_1IOc
9RTjrZxv
57


In [48]:
chain = (
    {"context": retriever,
      "question": RunnablePassthrough()}
         | prompt 
         | llm 
         | output_parser
     )

In [44]:
chain.invoke("How do we train transformers?")

AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}